# Voice Forge — Dataset Inspection & Tokenization Analysis
This notebook loads the training dataset generated by `scripts/build_jsonl.py`, applies the Mistral chat template, analyzes the sequence token lengths using the Mistral-7B tokenizer, checks for truncation risks, and performs vocabulary spot checks on key brand terms.

## 1. Imports and Environment Setup

In [ ]:
import os
import json
import random
import re
import matplotlib.pyplot as plt
from collections import Counter
from transformers import AutoTokenizer

## 2. Load Dataset & Print Breakdown
Load `data/dataset.jsonl` and print the total sample counts and content-type breakdown.

In [ ]:
# Determine correct path to dataset
dataset_path = "../data/dataset.jsonl"
if not os.path.exists(dataset_path):
    dataset_path = "data/dataset.jsonl"

dataset = []
with open(dataset_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            dataset.append(json.loads(line))

print(f"Total training pairs loaded: {len(dataset)}")

# Function to infer content type from the generated instruction text
def get_content_type(sample):
    instr = sample["instruction"].lower()
    if "how-to" in instr or "tutorial" in instr or "guide" in instr or "educational" in instr:
        return "how-to"
    elif "announcement" in instr or "launch" in instr or "release" in instr:
        return "announcement"
    elif "culture" in instr or "story" in instr or "reflecting" in instr or "experience" in instr:
        return "culture"
    else:
        return "product"

content_types = [get_content_type(s) for s in dataset]
breakdown = Counter(content_types)
print("\nContent-Type Breakdown:")
for ct, count in breakdown.items():
    print(f"  - {ct}: {count} ({count / len(dataset) * 100:.1f}%)")

## 3. Apply Mistral Chat Template
Format each pair into the format expected by Mistral-7B-Instruct:
`<s>[INST] {instruction}\n{input} [/INST] {output} </s>`

In [ ]:
formatted_samples = []
for s in dataset:
    formatted = f"<s>[INST] {s['instruction']}\n{s['input']} [/INST] {s['output']} </s>"
    formatted_samples.append(formatted)

print(f"Applied chat template to {len(formatted_samples)} samples.")
print(f"Example formatted text:\n{formatted_samples[0]}")

## 4. Token Length Analysis & Histogram
Load the Mistral-7B tokenizer to measure sequence lengths. We compute sequence statistics and plot a distribution histogram, highlighting the 512-token limit (truncation boundary).

In [ ]:
print("Loading Mistral tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2")
print("Tokenizer loaded successfully.\n")

token_lengths = []
exceeding_indices = []

for idx, sample_text in enumerate(formatted_samples):
    tokens = tokenizer.encode(sample_text)
    length = len(tokens)
    token_lengths.append(length)
    if length > 512:
        exceeding_indices.append((idx, length))

# Calculate stats
mean_len = sum(token_lengths) / len(token_lengths)
sorted_lens = sorted(token_lengths)
median_len = sorted_lens[len(sorted_lens) // 2]
max_len = max(token_lengths)
under_256_pct = sum(1 for l in token_lengths if l < 256) / len(token_lengths) * 100
over_512_pct = len(exceeding_indices) / len(token_lengths) * 100

print(f"Mean Sequence Length: {mean_len:.2f} tokens")
print(f"Median Sequence Length: {median_len} tokens")
print(f"Max Sequence Length: {max_len} tokens")
print(f"% of samples under 256 tokens: {under_256_pct:.2f}%")
print(f"% of samples over 512 tokens: {over_512_pct:.2f}%")

In [ ]:
# Plot the length distribution histogram
plt.figure(figsize=(10, 6))
plt.hist(token_lengths, bins=40, color="orange", edgecolor="black", alpha=0.7)
plt.axvline(mean_len, color="red", linestyle="dashed", linewidth=1.5, label=f"Mean: {mean_len:.2f}")
plt.axvline(median_len, color="blue", linestyle="dashed", linewidth=1.5, label=f"Median: {median_len}")
plt.axvline(512, color="purple", linestyle="dotted", linewidth=2.0, label="SFT Limit (512 tokens)")

plt.title("Voice Forge — Token Length Distribution (Mistral Chat Format)", fontsize=14)
plt.xlabel("Sequence Length (Tokens)", fontsize=12)
plt.ylabel("Number of Samples", fontsize=12)
plt.legend(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

## 5. Truncation Impact Check
Check if any samples exceed the 512-token limit. Truncation cuts off the end of completion sequences, which negatively impacts brand voice training (teaching the model to write incomplete sentences).

In [ ]:
if exceeding_indices:
    print(f"⚠️ FLAG: {len(exceeding_indices)} samples exceed the 512-token sequence limit!")
    print("Showing truncation details for these samples:\n")
    for idx, length in exceeding_indices:
        tokens = tokenizer.encode(formatted_samples[idx])
        kept_tokens = tokens[:512]
        truncated_tokens = tokens[512:]
        
        kept_text = tokenizer.decode(kept_tokens)
        truncated_text = tokenizer.decode(truncated_tokens)
        
        print(f"--- Sample {idx} (Length: {length} tokens) ---")
        print(f"Kept suffix text:\n... {kept_text[-150:]}")
        print(f"\n❌ TRUNCATED TEXT:\n{truncated_text}")
        print("=" * 60 + "\n")
else:
    print("✅ Success: No samples exceed the 512-token limit. No truncation will occur during SFT.")

## 6. Vocabulary Spot Check
Check how key brand-specific terms are tokenized by the Mistral tokenizer. If a term is split into too many sub-tokens, the model might struggle to learn its association, whereas single or double token values are ideal.

In [ ]:
words_to_check = ["Notion", "workspace", "block", "collaborate", "template"]

print("Vocabulary Tokenization Check:")
print("-" * 70)
print(f"{'Word':<15} | {'Token IDs':<20} | {'Sub-tokens Count':<18} | {'Decoded sub-tokens'}")
print("-" * 70)

for word in words_to_check:
    token_ids = tokenizer.encode(word, add_special_tokens=False)
    sub_tokens = [tokenizer.decode([tid]) for tid in token_ids]
    print(f"{word:<15} | {str(token_ids):<20} | {len(token_ids):<18} | {sub_tokens}")
print("-" * 70)

## 7. Sample Viewer
Display 3 random formatted samples from the dataset to inspect formatting, spacing, and templates.

In [ ]:
random.seed(42)  # For reproducibility
viewer_samples = random.sample(formatted_samples, 3)

for i, sample in enumerate(viewer_samples, 1):
    print(f"\nSample {i} of 3:")
    print("─" * 60)
    print(sample)
    print("─" * 60)